In [ ]:
%pip install rembg onnxruntime pillow mediapipe

### ЕСЛИ БИБЛИОТЕКА НЕ РАБОТАЕТ,ТО НАДО УДАЛИТЬ ЕЕ И УСТАНОВИТЬ СНОВА, НО ДРУГУЮ ВЕРСИЮ

In [ ]:
%pip uninstall mediapipe



In [ ]:
%pip install mediapipe==0.10.14

In [4]:
# hands_gif_overlay_two_hands.py
# pip install opencv-python mediapipe pillow

import time
import cv2
import numpy as np
from PIL import Image
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision


def load_gif_frames(path):
    """Читает PNG/GIF в список numpy-кадров RGBA."""
    img = Image.open(path)
    frames = []
    try:
        while True:
            frames.append(np.array(img.convert("RGBA")))
            img.seek(img.tell() + 1)
    except EOFError:
        pass
    if not frames:
        frames = [np.array(Image.open(path).convert("RGBA"))]
    return frames


LEFT_FRAMES = load_gif_frames("1115(2)_clean.png")
RIGHT_FRAMES = load_gif_frames("1115(4)_clean.png")
FRAME_TIME = 0.1  # секунды на кадр анимации

# --- новый API MediaPipe ---
options = vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,                      # <-- теперь распознаём две руки
    min_hand_detection_confidence=0.6,
)
landmarker = vision.HandLandmarker.create_from_options(options)


def overlay(frame, rgba, center, size=200):
    h, w = rgba.shape[:2]
    scale = size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    rgba = cv2.resize(rgba, (new_w, new_h), interpolation=cv2.INTER_AREA)

    x, y = int(center[0] - new_w / 2), int(center[1] - new_h / 2)
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(frame.shape[1], x + new_w), min(frame.shape[0], y + new_h)
    if x1 >= x2 or y1 >= y2:
        return frame

    patch = rgba[y1 - y:y2 - y, x1 - x:x2 - x]
    alpha = patch[:, :, 3:4] / 255.0
    roi = frame[y1:y2, x1:x2]
    frame[y1:y2, x1:x2] = (patch[:, :, :3] * alpha + roi * (1 - alpha)).astype(np.uint8)
    return frame


cap = cv2.VideoCapture(0)

# отдельный счётчик кадров анимации на каждую руку
frame_idx = {"Left": 0, "Right": 0}
last_switch = {"Left": time.time(), "Right": time.time()}
timestamp_ms = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp_ms += 33  # ~30 fps, метка должна монотонно расти

    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    if result.hand_landmarks:
        for lm, handed in zip(result.hand_landmarks, result.handedness):
            label = handed[0].category_name  # "Left" / "Right"

            wrist, middle = lm[0], lm[9]
            cx = int((wrist.x + middle.x) / 2 * frame.shape[1])
            cy = int((wrist.y + middle.y) / 2 * frame.shape[0])

            frames = LEFT_FRAMES if label == "Left" else RIGHT_FRAMES

            # у каждой руки свой таймер переключения кадров
            if time.time() - last_switch[label] > FRAME_TIME:
                frame_idx[label] = (frame_idx[label] + 1) % len(frames)
                last_switch[label] = time.time()

            frame = overlay(frame, frames[frame_idx[label]], (cx, cy))

            cv2.putText(frame, label.upper(), (cx - 40, cy - 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("Hands GIF", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 